In [2]:
%reload_ext autoreload
%autoreload 2 --print

In [16]:
import pandas as pd
from src.paths import TRANSFORMED_DATA_DIR

df= pd.read_parquet(TRANSFORMED_DATA_DIR/"tabular_data.parquet")
df

,rides_previous_672_hour,rides_previous_671_hour,rides_previous_670_hour,rides_previous_669_hour,rides_previous_668_hour,rides_previous_667_hour,rides_previous_666_hour,rides_previous_665_hour,rides_previous_664_hour,rides_previous_663_hour,...,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour,pickup_hour,location_id,target_rides_next_hour
0,2.0,3.0,2.0,2.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,2026-01-29,3,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,2.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2026-01-30,3,0.0
2,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,0.0,0.0,...,0.0,2.0,2.0,0.0,0.0,1.0,0.0,2026-01-31,3,0.0
3,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,2.0,2026-02-01,3,0.0
4,0.0,0.0,0.0,0.0,1.0,1.0,2.0,2.0,1.0,2.0,...,0.0,3.0,2.0,1.0,0.0,0.0,0.0,2026-02-02,3,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64185,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-26,199,0.0
64186,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-27,199,0.0
64187,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-28,199,0.0
64188,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2026-09-29,199,0.0


In [19]:
from datetime import datetime
from src.data_split import train_test_split

X_train, y_train, X_test, y_test = train_test_split(
    df,
    cutoff_date= datetime(2026, 7, 1, 0, 0),
    target_column_name= "target_rides_next_hour"

)

print(f"{X_train.shape=}")
print(f"{X_test.shape=}")
print(f"{y_train.shape=}")
print(f"{y_test.shape=}")



X_train.shape=(40086, 674)
X_test.shape=(24104, 674)
y_train.shape=(40086,)
y_test.shape=(24104,)


In [ ]:
# our first baseline model is very simple, not an ML model actually.
#  it just looks at the last demand at the previous hour and predicts next hour demand is the same as previous hour.
import numpy as np

class BaseLineModelPreviousHour:

    """
    Prediction= acrual demand at previous hour
    """
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        pass

    def predict(self, X_test: pd.DataFrame) -> np.array:
        return X_test[f"rides_previous_1_hour"]


In [20]:
model= BaseLineModelPreviousHour()
predictions= model.predict(X_test)
predictions


0        0.0
1        0.0
2        0.0
3        0.0
4        1.0
        ... 
24099    0.0
24100    0.0
24101    0.0
24102    0.0
24103    0.0
Name: rides_previous_1_hour, Length: 24104, dtype: float32

In [ ]:
from sklearn.metrics import mean_absolute_error
test_mae= mean_absolute_error(y_test, predictions)
print(f"{test_mae= :.4f}")

test_mae= 7.6043


In [ ]:
# second baseline model, we change the seasonality from 1 hour to 1 week
# meaning the prediction for next hour demand is the same as previous week same hour demand.

class BaseLineModelPreviousWeek:

    """
    Prediction= acrual demand at previous week same hour
    """
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        pass

    def predict(self, X_test: pd.DataFrame) -> np.array:
        return X_test[f"rides_previous_{7*24}_hour"]

In [30]:
model= BaseLineModelPreviousWeek()
predictions= model.predict(X_test)

In [28]:
test_mae= mean_absolute_error(y_test, predictions)
print(f"{test_mae= :.4f}")

test_mae= 5.3021


In [ ]:
# for the third baseline model, we get the intuition by comparing the previous 2 models, the change in seasonality affects model performance.
# so what if we use the historical demand and average them over 7, 14, 21 or 28 days ago?
class BaseLineModelLast4Weeks:

    """
    Prediction= actual demand observed averaged over 28 days ago
    """
    def fit(self, X_train: pd.DataFrame, y_train: pd.Series):
        pass

    def predict(self, X_test: pd.DataFrame) -> np.array:
        return 0.25 * (
            X_test[f"rides_previous_{7*24}_hour"] +\
            X_test[f"rides_previous_{2*7*24}_hour"] +\
            X_test[f"rides_previous_{3*7*24}_hour"] +\
            X_test[f"rides_previous_{4*7*24}_hour"]
        ) 

In [31]:
model= BaseLineModelLast4Weeks()
predictions= model.predict(X_test)

In [32]:
test_mae= mean_absolute_error(y_test, predictions)
print(f"{test_mae= :.4f}")

test_mae= 4.7825
